In [0]:
CREATE OR REPLACE TABLE production.supply_analytics.tour_feature_audit AS

WITH tour_option_features AS (
  SELECT 
    tour_id,
    tour_option_id,
    supplier_id,
    
    -- API Integration flags
    MAX(uses_prices_over_api) as uses_prices_over_api,
    MAX(uses_live_availability_and_price) as uses_live_availability_and_price,
    MAX(uses_pricing_scales_over_api) as uses_pricing_scales_over_api,
    MAX(uses_external_pricing) as uses_external_pricing,
    
    -- Feature flags
    MAX(has_addons_configured) as has_addons_configured,
    MAX(has_individual_pricing) as has_individual_pricing,
    MAX(has_group_pricing) as has_group_pricing,
    MAX(has_any_scale_pricing) as has_any_scale_pricing,
    MAX(has_scale_pricing_individual) as has_scale_pricing_individual,
    MAX(has_scale_pricing_group) as has_scale_pricing_group,
    MAX(has_scale_pricing_addons) as has_scale_pricing_addons,
    
    -- Operational features
    MAX(has_cutoff_configured) as has_cutoff_configured,
    MAX(has_min_participant_requirement) as has_min_participant_requirement,
    MAX(has_max_participant_limit) as has_max_participant_limit,
    MAX(has_capacity_limits) as has_capacity_limits,
    
    -- Category and tier counts
    AVG(num_individual_categories) as avg_num_individual_categories,
    AVG(num_group_categories) as avg_num_group_categories,
    AVG(num_addons) as avg_num_addons,
    AVG(total_individual_price_tiers) as avg_individual_tiers,
    AVG(total_group_price_tiers) as avg_group_tiers,
    AVG(total_addon_price_tiers) as avg_addon_tiers,
    AVG(total_price_tiers) as avg_total_price_tiers,
    
    -- Operational metrics
    AVG(pricing_dates_covered_next_12mo) as avg_dates_covered,
    AVG(pct_dates_with_vacancy_info) as avg_pct_vacancy_coverage
    
  FROM production.supply_analytics.feature_audit_base
  GROUP BY tour_id, tour_option_id, supplier_id
)

SELECT 
  t.tour_id,
  t.supplier_id,
  t.tour_title,
  t.location_name,
  t.activity_type,
  t.tour_category,
  t.date_of_creation,
  t.date_of_first_online,
  
  -- Tour structure
  COUNT(DISTINCT tof.tour_option_id) as num_tour_options,
  
  -- API Integration (if ANY tour option uses it, the tour uses it)
  MAX(tof.uses_prices_over_api) as uses_prices_over_api,
  MAX(tof.uses_live_availability_and_price) as uses_live_availability_and_price,
  MAX(tof.uses_pricing_scales_over_api) as uses_pricing_scales_over_api,
  MAX(tof.uses_external_pricing) as uses_external_pricing,
  
  -- Feature adoption (if ANY tour option has it, the tour has it)
  MAX(tof.has_addons_configured) as has_addons_configured,
  MAX(tof.has_individual_pricing) as has_individual_pricing,
  MAX(tof.has_group_pricing) as has_group_pricing,
  MAX(tof.has_any_scale_pricing) as has_any_scale_pricing,
  MAX(tof.has_scale_pricing_individual) as has_scale_pricing_individual,
  MAX(tof.has_scale_pricing_group) as has_scale_pricing_group,
  MAX(tof.has_scale_pricing_addons) as has_scale_pricing_addons,
  
  -- Operational features
  MAX(tof.has_cutoff_configured) as has_cutoff_configured,
  MAX(tof.has_min_participant_requirement) as has_min_participant_requirement,
  MAX(tof.has_max_participant_limit) as has_max_participant_limit,
  MAX(tof.has_capacity_limits) as has_capacity_limits,
  
  -- Category counts (average across tour options)
  ROUND(AVG(tof.avg_num_individual_categories), 1) as avg_num_individual_categories,
  ROUND(AVG(tof.avg_num_group_categories), 1) as avg_num_group_categories,
  ROUND(AVG(tof.avg_num_addons), 1) as avg_num_addons,
  
  -- Tier counts (average across tour options)
  ROUND(AVG(tof.avg_individual_tiers), 1) as avg_individual_price_tiers,
  ROUND(AVG(tof.avg_group_tiers), 1) as avg_group_price_tiers,
  ROUND(AVG(tof.avg_addon_tiers), 1) as avg_addon_price_tiers,
  ROUND(AVG(tof.avg_total_price_tiers), 1) as avg_total_price_tiers,
  
  -- Operational metrics
  ROUND(AVG(tof.avg_dates_covered), 0) as avg_dates_covered,
  ROUND(AVG(tof.avg_pct_vacancy_coverage), 1) as avg_pct_vacancy_coverage,
  
  -- Feature count for this tour
  (MAX(tof.has_addons_configured) + 
   MAX(tof.has_group_pricing) + 
   MAX(tof.has_any_scale_pricing) + 
   MAX(tof.has_cutoff_configured) + 
   MAX(tof.has_capacity_limits)) as tour_feature_count

FROM (
  SELECT DISTINCT 
    tour_id, 
    supplier_id, 
    tour_title, 
    location_name, 
    activity_type, 
    tour_category,
    date_of_creation,
    date_of_first_online
  FROM production.supply_analytics.feature_audit_test
) t
LEFT JOIN tour_option_features tof ON t.tour_id = tof.tour_id
GROUP BY t.tour_id, t.supplier_id, t.tour_title, t.location_name, t.activity_type, 
         t.tour_category, t.date_of_creation, t.date_of_first_online;





-- COMPREHENSIVE EXECUTIVE SUMMARY - All Pricing Features with Categories and Tiers

SELECT 
  'MARKETPLACE OVERVIEW' as section,
  'Total Active Tours' as metric,
  COUNT(DISTINCT tour_id) as value,
  NULL as pct_of_tours
FROM production.supply_analytics.tour_feature_audit

UNION ALL
SELECT 
  'MARKETPLACE OVERVIEW',
  'Total Active Suppliers',
  COUNT(DISTINCT supplier_id),
  NULL
FROM production.supply_analytics.tour_feature_audit

UNION ALL
SELECT 
  'MARKETPLACE OVERVIEW',
  'Total Tour Options',
  SUM(num_tour_options),
  NULL
FROM production.supply_analytics.tour_feature_audit

UNION ALL
SELECT 
  'MARKETPLACE OVERVIEW',
  'Avg Tour Options per Tour',
  ROUND(AVG(num_tour_options), 1),
  NULL
FROM production.supply_analytics.tour_feature_audit

-- INDIVIDUAL PRICING
UNION ALL
SELECT 
  'INDIVIDUAL PRICING',
  'Tours with Individual Pricing',
  SUM(CAST(has_individual_pricing AS INT)),
  ROUND(100.0 * SUM(CAST(has_individual_pricing AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

UNION ALL
SELECT 
  'INDIVIDUAL PRICING',
  'Avg Individual Categories per Tour',
  ROUND(AVG(avg_num_individual_categories), 2),
  NULL
FROM production.supply_analytics.tour_feature_audit

UNION ALL
SELECT 
  'INDIVIDUAL PRICING',
  'Avg Individual Price Tiers per Tour',
  ROUND(AVG(avg_individual_price_tiers), 1),
  NULL
FROM production.supply_analytics.tour_feature_audit

UNION ALL
SELECT 
  'INDIVIDUAL PRICING',
  'Tours with Scale Pricing on Individual',
  SUM(CAST(has_scale_pricing_individual AS INT)),
  ROUND(100.0 * SUM(CAST(has_scale_pricing_individual AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

-- GROUP PRICING
UNION ALL
SELECT 
  'GROUP PRICING',
  'Tours with Group Pricing',
  SUM(CAST(has_group_pricing AS INT)),
  ROUND(100.0 * SUM(CAST(has_group_pricing AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

UNION ALL
SELECT 
  'GROUP PRICING',
  'Avg Group Categories per Tour',
  ROUND(AVG(avg_num_group_categories), 2),
  NULL
FROM production.supply_analytics.tour_feature_audit

UNION ALL
SELECT 
  'GROUP PRICING',
  'Avg Group Price Tiers per Tour',
  ROUND(AVG(avg_group_price_tiers), 1),
  NULL
FROM production.supply_analytics.tour_feature_audit

UNION ALL
SELECT 
  'GROUP PRICING',
  'Tours with Scale Pricing on Group',
  SUM(CAST(has_scale_pricing_group AS INT)),
  ROUND(100.0 * SUM(CAST(has_scale_pricing_group AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

-- ADDONS
UNION ALL
SELECT 
  'ADDONS',
  'Tours with Addons Configured',
  SUM(CAST(has_addons_configured AS INT)),
  ROUND(100.0 * SUM(CAST(has_addons_configured AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

UNION ALL
SELECT 
  'ADDONS',
  'Avg Number of Addons per Tour',
  ROUND(AVG(avg_num_addons), 2),
  NULL
FROM production.supply_analytics.tour_feature_audit

UNION ALL
SELECT 
  'ADDONS',
  'Avg Addon Price Tiers per Tour',
  ROUND(AVG(avg_addon_price_tiers), 1),
  NULL
FROM production.supply_analytics.tour_feature_audit

UNION ALL
SELECT 
  'ADDONS',
  'Tours with Scale Pricing on Addons',
  SUM(CAST(has_scale_pricing_addons AS INT)),
  ROUND(100.0 * SUM(CAST(has_scale_pricing_addons AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

-- SCALE/TIER PRICING SUMMARY
UNION ALL
SELECT 
  'SCALE PRICING',
  'Tours with Any Scale Pricing',
  SUM(CAST(has_any_scale_pricing AS INT)),
  ROUND(100.0 * SUM(CAST(has_any_scale_pricing AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

UNION ALL
SELECT 
  'SCALE PRICING',
  'Avg Total Price Tiers per Tour',
  ROUND(AVG(avg_total_price_tiers), 1),
  NULL
FROM production.supply_analytics.tour_feature_audit

-- API INTEGRATIONS
UNION ALL
SELECT 
  'API INTEGRATION',
  'Tours Using Prices Over API',
  SUM(CAST(uses_prices_over_api AS INT)),
  ROUND(100.0 * SUM(CAST(uses_prices_over_api AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

UNION ALL
SELECT 
  'API INTEGRATION',
  'Tours Using Live Availability and Price',
  SUM(CAST(uses_live_availability_and_price AS INT)),
  ROUND(100.0 * SUM(CAST(uses_live_availability_and_price AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

UNION ALL
SELECT 
  'API INTEGRATION',
  'Tours Using Pricing Scales Over API',
  SUM(CAST(uses_pricing_scales_over_api AS INT)),
  ROUND(100.0 * SUM(CAST(uses_pricing_scales_over_api AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

UNION ALL
SELECT 
  'API INTEGRATION',
  'Tours Using Any External Pricing',
  SUM(CAST(uses_external_pricing AS INT)),
  ROUND(100.0 * SUM(CAST(uses_external_pricing AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

-- OPERATIONAL FEATURES
UNION ALL
SELECT 
  'OPERATIONAL FEATURES',
  'Tours with Cutoff Time Configured',
  SUM(CAST(has_cutoff_configured AS INT)),
  ROUND(100.0 * SUM(CAST(has_cutoff_configured AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

UNION ALL
SELECT 
  'OPERATIONAL FEATURES',
  'Tours with Capacity Limits',
  SUM(CAST(has_capacity_limits AS INT)),
  ROUND(100.0 * SUM(CAST(has_capacity_limits AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

UNION ALL
SELECT 
  'OPERATIONAL FEATURES',
  'Tours with Min Participant Requirement',
  SUM(CAST(has_min_participant_requirement AS INT)),
  ROUND(100.0 * SUM(CAST(has_min_participant_requirement AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

UNION ALL
SELECT 
  'OPERATIONAL FEATURES',
  'Tours with Max Participant Limit',
  SUM(CAST(has_max_participant_limit AS INT)),
  ROUND(100.0 * SUM(CAST(has_max_participant_limit AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

-- OPERATIONAL HEALTH
UNION ALL
SELECT 
  'OPERATIONAL HEALTH',
  'Avg Date Coverage (Days in Next 12 Months)',
  ROUND(AVG(avg_dates_covered), 0),
  NULL
FROM production.supply_analytics.tour_feature_audit

UNION ALL
SELECT 
  'OPERATIONAL HEALTH',
  'Avg Vacancy Coverage (% of Dates)',
  ROUND(AVG(avg_pct_vacancy_coverage), 1),
  NULL
FROM production.supply_analytics.tour_feature_audit

-- FEATURE SOPHISTICATION
UNION ALL
SELECT 
  'FEATURE SOPHISTICATION',
  'Tours with 0 Advanced Features',
  SUM(CASE WHEN tour_feature_count = 0 THEN 1 ELSE 0 END),
  ROUND(100.0 * SUM(CASE WHEN tour_feature_count = 0 THEN 1 ELSE 0 END) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

UNION ALL
SELECT 
  'FEATURE SOPHISTICATION',
  'Avg Feature Count per Tour (0-5)',
  ROUND(AVG(tour_feature_count), 2),
  NULL
FROM production.supply_analytics.tour_feature_audit

ORDER BY section, metric;

